In [2]:
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

MODEL_DIR = "/home/ubuntu/FairyTaleSL/model"
if MODEL_DIR not in sys.path:
    sys.path.insert(0, MODEL_DIR)

from data_ksl2 import (
    INPUT_NPZ_DIR,
    to_30fps,
    drop_dummy_frames,
    crop_video_by_hand_detection,
    interpolate_short_gaps,
    remove_short_hand_appearances,
    person_center_scale,
    gaussian_noise,
)

sns.set_theme(style="whitegrid")


In [1]:
from typing import Any, Dict, Union
import importlib.util

CONFIG_PATH = "/home/ubuntu/FairyTaleSL/model/configs/cnn1d_mediapipe_sign_without_face.py"

spec = importlib.util.spec_from_file_location("cnn1d_cfg", CONFIG_PATH)
cnn1d_cfg = importlib.util.module_from_spec(spec)
spec.loader.exec_module(cnn1d_cfg)
RANDOM_HORIZONTAL_FLIP_CFG = getattr(cnn1d_cfg, "RANDOM_HORIZONTAL_FLIP", None)

DEFAULT_POSE_LEFT_RIGHT_PAIRS = (
    (1, 4),
    (2, 5),
    (3, 6),
    (7, 8),
    (9, 10),
    (11, 12),
    (13, 14),
    (15, 16),
    (17, 18),
    (19, 20),
    (21, 22),
)
DEFAULT_LEFT_HAND_RANGE = (23, 44)
DEFAULT_RIGHT_HAND_RANGE = (44, 65)


def random_horizontal_flip_keypoints(
    keypoint: np.ndarray,
    flip_cfg: Union[Dict[str, Any], bool, None],
    test_mode: bool = False,
) -> np.ndarray:
    if not flip_cfg:
        return keypoint
    if flip_cfg is True:
        flip_cfg = {}
    if not isinstance(flip_cfg, dict):
        raise TypeError("horizontal flip config must be a dict, bool, or None.")
    if not flip_cfg.get("enabled", True):
        return keypoint
    if test_mode and not flip_cfg.get("apply_in_test", False):
        return keypoint
    if keypoint.ndim != 4:
        raise ValueError(f"keypoint must have shape (M, T, V, C), got {keypoint.shape}.")
    if keypoint.shape[-1] < 1:
        raise ValueError("horizontal flip requires at least an x channel.")

    probability = 1.0
    if probability <= 0.0 or np.random.random() >= probability:
        return keypoint

    x_channel = int(flip_cfg.get("x_channel", 0))
    if not (0 <= x_channel < keypoint.shape[-1]):
        raise ValueError(f"x_channel={x_channel} is out of range for C={keypoint.shape[-1]}.")

    out = keypoint.astype(np.float32, copy=True)
    x_max = float(flip_cfg.get("x_max", 1.0))
    x_min = float(flip_cfg.get("x_min", 0.0))
    out[..., x_channel] = x_max + x_min - out[..., x_channel]

    if not flip_cfg.get("swap_left_right", True):
        return out

    num_joints = out.shape[2]
    pairs = flip_cfg.get("left_right_pairs", DEFAULT_POSE_LEFT_RIGHT_PAIRS)
    for left, right in pairs:
        left = int(left)
        right = int(right)
        if 0 <= left < num_joints and 0 <= right < num_joints:
            out[:, :, [left, right], :] = out[:, :, [right, left], :]

    hand_ranges = flip_cfg.get(
        "hand_ranges",
        (DEFAULT_LEFT_HAND_RANGE, DEFAULT_RIGHT_HAND_RANGE),
    )
    if hand_ranges:
        left_range, right_range = hand_ranges
        left_start, left_end = [int(item) for item in left_range]
        right_start, right_end = [int(item) for item in right_range]
        left_len = left_end - left_start
        right_len = right_end - right_start
        if (
            left_len == right_len
            and left_len > 0
            and 0 <= left_start <= left_end <= num_joints
            and 0 <= right_start <= right_end <= num_joints
        ):
            left_hand = out[:, :, left_start:left_end, :].copy()
            out[:, :, left_start:left_end, :] = out[:, :, right_start:right_end, :]
            out[:, :, right_start:right_end, :] = left_hand

    return out


def apply_random_horizontal_flip_to_npz(data, flip_cfg, test_mode=False):
    pose = np.asarray(data["pose"], dtype=np.float32)[:, :23]
    left = np.asarray(data["left_hand"], dtype=np.float32)[:, :21]
    right = np.asarray(data["right_hand"], dtype=np.float32)[:, :21]
    keypoint = np.concatenate([pose, left, right], axis=1)[None]
    flipped = random_horizontal_flip_keypoints(keypoint, flip_cfg, test_mode=test_mode)[0]

    out = {key: data[key].copy() for key in data.files}
    out["pose"] = np.asarray(out["pose"], dtype=np.float32).copy()
    out["left_hand"] = np.asarray(out["left_hand"], dtype=np.float32).copy()
    out["right_hand"] = np.asarray(out["right_hand"], dtype=np.float32).copy()
    out["pose"][:, :23] = flipped[:, :23]
    out["left_hand"][:, :21] = flipped[:, 23:44]
    out["right_hand"][:, :21] = flipped[:, 44:65]
    return out

print("RANDOM_HORIZONTAL_FLIP_CFG:", RANDOM_HORIZONTAL_FLIP_CFG)


NameError: name 'np' is not defined

In [17]:
sample_npz_file = "070.npz"
sample_npz_path = os.path.join(INPUT_NPZ_DIR, sample_npz_file)
summary_csv_path = os.path.join(INPUT_NPZ_DIR, "all_summary.csv")

try:
    raw_data = np.load(sample_npz_path, allow_pickle=True)
    print(f"Successfully loaded {sample_npz_path}")
except FileNotFoundError:
    raise FileNotFoundError(f"{sample_npz_path} not found. Please ensure the file exists.")

merged_df = pd.read_csv(summary_csv_path)
video_name = os.path.splitext(sample_npz_file)[0]
video_id = int(video_name)
video_df = merged_df[merged_df["video"] == video_id].copy()

print("raw:", raw_data["pose"].shape, video_df.shape)

raw_data, video_df = to_30fps(raw_data, video_df, front=True)
print("30fps:", raw_data["pose"].shape, video_df.shape)

raw_data, video_df = drop_dummy_frames(raw_data, video_df, threshold=0.8)
print("drop dummy:", raw_data["pose"].shape, video_df.shape)

raw_data, video_df = crop_video_by_hand_detection(raw_data, video_df)
if raw_data is None or video_df is None:
    raise ValueError(f"No valid crop interval found for video {video_name}.")
print("crop:", raw_data["pose"].shape, video_df.shape)

raw_data, video_df = interpolate_short_gaps(raw_data, video_df, max_gap=10)
print("interpolate:", raw_data["pose"].shape, video_df.shape)

raw_data, video_df = remove_short_hand_appearances(raw_data, video_df, max_appear=10)
print("remove_short_hand_appearances:", raw_data["pose"].shape, video_df.shape)

raw_data, video_df = person_center_scale(
raw_data,
video_df,
scale=1.7, #1.7, 1.5, 1.3
)

raw_data, video_df = gaussian_noise(raw_data, video_df, scale=0.001) # 0.002/0.001/0.0005
print("gaussian:", raw_data["pose"].shape, video_df.shape)

# 아래 시각화 셀에서 사용할 최종 결과
data = raw_data
processed_df = video_df


Successfully loaded /home/ubuntu/dataset/holistic_result_comp2/070.npz
raw: (161, 33, 4) (161, 7)
30fps: (81, 33, 4) (81, 7)
drop dummy: (65, 33, 4) (65, 7)
crop: (39, 33, 4) (39, 8)
interpolate: (39, 33, 4) (39, 8)
remove_short_hand_appearances: (39, 33, 4) (39, 8)
gaussian: (39, 33, 4) (39, 8)


In [ ]:
from matplotlib import animation, rcParams
from IPython.display import HTML, display

INPUT_NPZ_DIR = "/home/ubuntu/dataset/holistic_result_comp2_augmented_20_vari_3/test"

sample_npz_file = "10_69.npz"
sample_npz_path = os.path.join(INPUT_NPZ_DIR, sample_npz_file)
data = np.load(sample_npz_path, allow_pickle=True)
#data = apply_random_horizontal_flip_to_npz(data, RANDOM_HORIZONTAL_FLIP_CFG, test_mode=False)
# Get the total number of frames for the processed video
num_frames = data["pose"].shape[0] if "pose" in data else 0
if num_frames == 0:
    raise ValueError("data에 pose frame이 없습니다. 전처리 셀을 먼저 실행했는지 확인하세요.")

print(f"Frames to visualize: {num_frames}")

body_part_keys_to_plot = {
    "Pose": "pose",
    "Left Hand": "left_hand",
    "Right Hand": "right_hand",
}

body_part_colors = {
    "Pose": "blue",
    "Left Hand": "green",
    "Right Hand": "red",
}

# 모든 프레임에서 동일한 축 범위를 사용합니다.
fixed_xlim = (0, 1)
fixed_ylim = (1.2, 0)  # y축 반전: 화면 좌표처럼 위가 0, 아래가 1.2


def get_frame_points(frame_idx):
    points_by_part = {}

    for name, key in body_part_keys_to_plot.items():
        arr_for_frame = data[key][frame_idx]

        # pose는 face/hand와 겹치는 뒤쪽 landmark를 제외하고 body 쪽만 확인
        if key == "pose":
            arr_for_frame = arr_for_frame[:23]

        valid_points_mask = arr_for_frame[:, 3] > 0
        valid_points = arr_for_frame[valid_points_mask]
        points_by_part[name] = valid_points[:, :2] if valid_points.size > 0 else np.empty((0, 2))

    return points_by_part

fig, ax = plt.subplots(1, 1, figsize=(10, 8))
scatters = {}

for name, color in body_part_colors.items():
    scatters[name] = ax.scatter(
        [],
        [],
        s=20,
        alpha=0.8,
        edgecolors="w",
        linewidths=0.5,
        label=name,
        color=color,
    )

ax.set_xlim(*fixed_xlim)
ax.set_ylim(*fixed_ylim)
ax.set_xlabel("X Coordinate")
ax.set_ylabel("Y Coordinate")
ax.legend(title="Body Part")
ax.grid(True, linestyle="--", alpha=0.6)
title = ax.set_title("")


def update(frame_idx):
    points_by_part = get_frame_points(frame_idx)

    for name, offsets in points_by_part.items():
        scatters[name].set_offsets(offsets)

    ax.set_xlim(*fixed_xlim)
    ax.set_ylim(*fixed_ylim)
    title.set_text(f"Combined Landmark Visualization for Frame {frame_idx} from {sample_npz_file}")
    return [title, *scatters.values()]

fps = float(data.get("fps", 30))
fps = max(1, min(fps, 30))
interval = 1000 / fps

ani = animation.FuncAnimation(
    fig,
    update,
    frames=num_frames,
    interval=interval,
    blit=False,
    repeat=True,
)

# 파일 저장 없이 셀 안에서 바로 재생합니다.
rcParams["animation.embed_limit"] = 100
html = ani.to_jshtml(fps=fps, default_mode="loop")
plt.close(fig)
display(HTML(html))


Frames to visualize: 84


In [55]:
from matplotlib import animation, rcParams
from IPython.display import HTML, display

INPUT_NPZ_DIR = "/home/ubuntu/dataset/holistic_result_comp2_augmented_20_vari_3/train"

sample_npz_file = "17_166.npz"
sample_npz_path = os.path.join(INPUT_NPZ_DIR, sample_npz_file)
data = np.load(sample_npz_path, allow_pickle=True)
data = apply_random_horizontal_flip_to_npz(data, RANDOM_HORIZONTAL_FLIP_CFG, test_mode=False)
# Get the total number of frames for the processed video
num_frames = data["pose"].shape[0] if "pose" in data else 0
if num_frames == 0:
    raise ValueError("data에 pose frame이 없습니다. 전처리 셀을 먼저 실행했는지 확인하세요.")

print(f"Frames to visualize: {num_frames}")

body_part_keys_to_plot = {
    "Pose": "pose",
    "Left Hand": "left_hand",
    "Right Hand": "right_hand",
}

body_part_colors = {
    "Pose": "blue",
    "Left Hand": "green",
    "Right Hand": "red",
}

# 모든 프레임에서 동일한 축 범위를 사용합니다.
fixed_xlim = (0, 1)
fixed_ylim = (1.2, 0)  # y축 반전: 화면 좌표처럼 위가 0, 아래가 1.2


def get_frame_points(frame_idx):
    points_by_part = {}

    for name, key in body_part_keys_to_plot.items():
        arr_for_frame = data[key][frame_idx]

        # pose는 face/hand와 겹치는 뒤쪽 landmark를 제외하고 body 쪽만 확인
        if key == "pose":
            arr_for_frame = arr_for_frame[:23]

        valid_points_mask = arr_for_frame[:, 3] > 0
        valid_points = arr_for_frame[valid_points_mask]
        points_by_part[name] = valid_points[:, :2] if valid_points.size > 0 else np.empty((0, 2))

    return points_by_part

fig, ax = plt.subplots(1, 1, figsize=(10, 8))
scatters = {}

for name, color in body_part_colors.items():
    scatters[name] = ax.scatter(
        [],
        [],
        s=20,
        alpha=0.8,
        edgecolors="w",
        linewidths=0.5,
        label=name,
        color=color,
    )

ax.set_xlim(*fixed_xlim)
ax.set_ylim(*fixed_ylim)
ax.set_xlabel("X Coordinate")
ax.set_ylabel("Y Coordinate")
ax.legend(title="Body Part")
ax.grid(True, linestyle="--", alpha=0.6)
title = ax.set_title("")


def update(frame_idx):
    points_by_part = get_frame_points(frame_idx)

    for name, offsets in points_by_part.items():
        scatters[name].set_offsets(offsets)

    ax.set_xlim(*fixed_xlim)
    ax.set_ylim(*fixed_ylim)
    title.set_text(f"Combined Landmark Visualization for Frame {frame_idx} from {sample_npz_file}")
    return [title, *scatters.values()]

fps = float(data.get("fps", 30))
fps = max(1, min(fps, 30))
interval = 1000 / fps

ani = animation.FuncAnimation(
    fig,
    update,
    frames=num_frames,
    interval=interval,
    blit=False,
    repeat=True,
)

# 파일 저장 없이 셀 안에서 바로 재생합니다.
rcParams["animation.embed_limit"] = 100
html = ani.to_jshtml(fps=fps, default_mode="loop")
plt.close(fig)
display(HTML(html))


Frames to visualize: 65
